In [1]:
print("hello world2")

hello world2


In [2]:
from datasets import Dataset
from datasets import DatasetInfo
import numpy as np
import torch
class SpectrogramDataset:
    def __init__(self, spectrograms, labels, mean=None, std=None):
        """
        spectrograms: List or Array of shape (N, Time, Freq=128)
        labels: List or Array of shape (N,)
        """
        self.spectrograms = spectrograms
        self.labels = labels
        
        # AST Normalization (Crucial step from the paper)
        # If mean/std are not provided, calculate them (like get_norm_stats.py)
        if mean is None or std is None:
            self.mean = np.mean(spectrograms)
            self.std = np.std(spectrograms)
        else:
            self.mean = mean
            self.std = std

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        spec = self.spectrograms[idx]
        
        # 1. Normalize: (input - mean) / (std * 2) 
        # The *2 is specific to the AST paper to bring data roughly between -0.5 and 0.5
        norm_spec = (spec - self.mean) / (self.std * 2)
        
        # 2. Convert to Tensor
        norm_spec = torch.tensor(norm_spec, dtype=torch.float32)
        
        # 3. Transpose if necessary? 
        # HuggingFace AST expects (Batch, Time, Freq). 
        # Your kaldi.fbank likely outputs (Time, Freq), which is correct.
        
        return {"input_values": norm_spec, "labels": self.labels[idx]}

    # Helper to convert to HF Dataset
    def create_hf_dataset(self):
        # norm_specs = [(s - self.mean) / (self.std * 2) for s in self.spectrograms]
        info = DatasetInfo(
            description=f"Spectrogram dataset mean {self.mean} , std: {self.std}"
        )
        ds = Dataset.from_dict({
            "input_values": self.spectrograms,
            "label": self.labels,
        },info=info)



        return ds

In [15]:
import matplotlib.pyplot as plt
import torch
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from IPython.display import clear_output
from tqdm import tqdm
# 1. Setup
# ---------------------------------------------------------
repo_id = "" # Update this if needed (or keep empty if local)
local_root = "/home/rtalwar/robot-imitation-glue/datasets/red_round_button_small_train"
# Target repo to save the new dataset (can be same as source if you have write access)
target_repo_id = "ramen-noodels/audio_red_round_button_small_train_unnormalized"
# ... (your existing setup code)
dataset = LeRobotDataset(repo_id=repo_id, root=local_root)
n_episodes = dataset.meta.total_episodes

print(f"Total episodes: {n_episodes}")
print(f"Total frames: {len(dataset)}")

# Calculate lengths if removing the last 25, 50, and 75 episodes
for drop_count in [0,25, 50, 75]:
    remaining_episodes = n_episodes - drop_count
    if remaining_episodes > 0:
        # The total frames of the first 'remaining_episodes' is the 'to' index of the last retained episode
        new_len = dataset.meta.episodes[remaining_episodes - 1]["dataset_to_index"]
        print(f"Length if you remove the last {drop_count} episodes: {new_len}")
    else:
        print(f"Cannot remove {drop_count} episodes because the dataset only has {n_episodes} episodes.")

Total episodes: 100
Total frames: 9559
Length if you remove the last 0 episodes: 9559
Length if you remove the last 25 episodes: 7061
Length if you remove the last 50 episodes: 4697
Length if you remove the last 75 episodes: 2222


In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from IPython.display import clear_output
from tqdm import tqdm

# 1. Setup
# ---------------------------------------------------------
repo_id = "" # Update this if needed (or keep empty if local)
local_root = "/home/rtalwar/robot-imitation-glue/datasets/red_round_button_small_train"
# Target repo to save the new dataset (can be same as source if you have write access)
target_repo_id = "ramen-noodels/audio_red_round_button_small_train_unnormalized"
dataset = LeRobotDataset(repo_id=repo_id, root=local_root)
n_episodes = dataset.meta.total_episodes

# This list will hold the final labels for the entire dataset
all_labels = [0] * len(dataset)

# 2. Generate Heuristic Labels
# ---------------------------------------------------------
print("Generating heuristic labels...")
for episode_idx in tqdm(range(n_episodes)):
    # Get indices for this episode
    episode_indices = dataset.meta.episodes[episode_idx]
    episode_start_idx = episode_indices["dataset_from_index"]
    episode_to_idx = episode_indices["dataset_to_index"]
    
    # Default state for start of episode
    current_label = 1 
    lastclick_frame_idx = -1 
    
    # Iterate through this specific episode
    first_press=True
    for i in range(episode_start_idx, episode_to_idx):
        frame = dataset[i]
        
        # YOUR HEURISTIC LOGIC
        # --------------------
        # Logic: Label is 1 (visible/default). 
        # If btn pressed (0), label becomes 0 (not visible/clicked).
        # 30 frames later, it becomes 1 again.
        
        if frame["btn_state"] == 0:
            current_label = 0
            lastclick_frame_idx = i
        # if frame["btn_state"] == 0:
        #     if first_press:
        #         first_press = False
        #         first_click_frame_idx = i
        #     if i >= first_click_frame_idx + 5:
        #         current_label = 0
        #     lastclick_frame_idx = i
        # else:
        #     first_press = True
        #     current_label = 1
        
        if lastclick_frame_idx != -1 and i >= lastclick_frame_idx + 30:
            current_label = 1
            # Reset click frame so we don't trigger this immediately again 
            # unless button is pressed again
            lastclick_frame_idx = -1 

        all_labels[i] = current_label

print("Heuristic labels generated.")

Generating heuristic labels...


100%|██████████| 25/25 [00:13<00:00,  1.80it/s]

Heuristic labels generated.


In [8]:
# save all_labels as pickle
import pickle
with open("all_labels.pkl", "wb") as f:
    pickle.dump(all_labels, f)

In [9]:
#load all_labels from pickle
import pickle
with open("all_labels.pkl", "rb") as f:
    all_labels = pickle.load(f) 

In [10]:
# 3. Interactive Verification
# ---------------------------------------------------------
def plot_transition(center_idx, labels, dataset):
    """Plots 4 frames before and 4 after the center index."""
    # Define window (handle dataset boundaries)
    start = max(0, center_idx - 4)
    end = min(len(dataset), center_idx + 5) # +5 because range is exclusive at end
    
    indices = list(range(start, end))
    num_frames = len(indices)
    
    fig, axes = plt.subplots(1, num_frames, figsize=(20, 3))
    
    for ax_idx, ds_idx in enumerate(indices):
        frame = dataset[ds_idx]
        # [C, H, W] -> [H, W]
        spectrogram = frame["spectogram_values"][0, :, :]

        ax = axes[ax_idx]
        ax.imshow(spectrogram, aspect='auto')
        
        # Color code title: Red if label changed relative to previous frame
        is_transition = (ds_idx > 0 and labels[ds_idx] != labels[ds_idx-1])
        title_color = 'red' if is_transition else 'black'
        
        ax.set_title(f"Idx: {ds_idx}\nLbl: {labels[ds_idx]}\nsum: {torch.sum(spectrogram)}\nmax: {torch.max(spectrogram)}", color=title_color)
        ax.axis('off')
        
        # Highlight the center frame
        if ds_idx == center_idx:
            for spine in ax.spines.values():
                spine.set_edgecolor('blue')
                spine.set_linewidth(3)

    plt.tight_layout()
    plt.show() # Non-blocking so code continues to input
    return fig

def spectrogram_max(idx, dataset):
    frame = dataset[idx]
    spectrogram = frame["spectogram_values"][0, :, :]
    return torch.max(spectrogram)

# Find indices where label changes
transitions = []
for i in range(1, len(all_labels)):
    if all_labels[i] != all_labels[i-1]:
        transitions.append(i)

# Heuristic adjustment: for 1->0 transitions, enforce max >= 0.85
adjusted = 0
for t_idx in list(transitions):
    if t_idx + 1 >= len(all_labels):
        continue
    if all_labels[t_idx - 1] == 1 and all_labels[t_idx] == 0:
        curr_max = spectrogram_max(t_idx, dataset)
        if curr_max < 0.82:
            next_max = spectrogram_max(t_idx + 1, dataset)
            if next_max >= 0.82:
                all_labels[t_idx] = 1
                all_labels[t_idx + 1] = 0
                adjusted += 1

# Recompute transitions after heuristic adjustment
transitions = []
for i in range(1, len(all_labels)):
    if all_labels[i] != all_labels[i-1]:
        transitions.append(i)

print(f"Found {len(transitions)} label transitions.")
if adjusted > 0:
    print(f"Heuristic-shifted {adjusted} transitions by +1.")
print("Starting interactive verification...")

i = 0
while i < len(transitions):
    t_idx = transitions[i]
    print(f"\n--- Checking transition at index {t_idx} ({i+1}/{len(transitions)}) ---")
    print(f"Label went from {all_labels[t_idx-1]} to {all_labels[t_idx]}")
    if all_labels[t_idx-1] == 0 :
        i+=1
        continue
    
    fig = plot_transition(t_idx, all_labels, dataset)
    
    user_input = input(
        "[Enter]: Keep\n"
        "[num]: Shift transition to this index\n"
        "[f]: Flip label at this index\n"
        "[q]: Quit verification\n"
        "Selection: "
    )
    if i%2==0:
        plt.close(fig) # Close plot
        plt.close('all')
        clear_output(wait=True)  # removes old outputs from the notebook
    
    if user_input.lower() == 'q':
        break
    elif user_input == '':
        i += 1
        continue
    elif user_input.lower() == 'f':
        # Flip current label
        all_labels[t_idx] = 1 - all_labels[t_idx]
        print(f"Label at {t_idx} flipped to {all_labels[t_idx]}")
        # Don't increment 'i' so we can verify the change immediately
    elif user_input.isdigit():
        new_idx = int(user_input)
        # Shift logic: If user says transition is actually at 50, but we found it at 48.
        # We need to fill the gap with the 'previous' label.
        
        # Determine which label is the "pre-transition" label
        target_val = all_labels[t_idx] # The value it changed TO
        prev_val = all_labels[t_idx-1] # The value it changed FROM
        
        # Apply corrections
        # This is a simplified logic: it forces a split at the new index
        # You might need to adjust logic depending on if you are shifting left or right
        if new_idx > t_idx:
            # Shift transition RIGHT: indices between t_idx and new_idx become prev_val
            for k in range(t_idx, new_idx):
                all_labels[k] = prev_val
        elif new_idx < t_idx:
            # Shift transition LEFT: indices between new_idx and t_idx become target_val
            for k in range(new_idx, t_idx):
                all_labels[k] = target_val
        
        print(f"Transition shifted to {new_idx}")
        # Update transition list for future iterations if necessary, 
        # or just re-verify this section
        if new_idx not in transitions:
            transitions[i] = new_idx 
            # We stay at 'i' to verify the new look
    else:
        print("Invalid input, skipping.")
        i += 1
# 4. Save to Hugging Face Hub
# ---------------------------------------------------------
print("\nPreparing to save...")

# # Get the underlying Hugging Face dataset object
# hf_dataset = dataset.hf_dataset

# # Add the new labels as a column
# # We remove the column if it exists to overwrite, or just add it
# if "observation.labels" in hf_dataset.column_names:
#     hf_dataset = hf_dataset.remove_columns("observation.labels")

# # Add the new column
# hf_dataset = hf_dataset.add_column("observation.labels", all_labels)

# # Push to hub
# print(f"Pushing to {target_repo_id}...")
# hf_dataset.push_to_hub(target_repo_id)

# print("Done! Dataset saved.")
spectograms=[dataset[i]["spectogram_values"][0,:,:] for i in range(0,len(dataset))] 
spec_ds=SpectrogramDataset(spectograms,all_labels)
spec_ds_hf = spec_ds.create_hf_dataset()
spec_ds_hf.push_to_hub(target_repo_id)


Preparing to save...


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/ramen-noodels/audio_red_round_button_small_val_unnormalized/commit/af4133fc73213dc61a2eb123052dd0fca6f05157', commit_message='Upload dataset', commit_description='', oid='af4133fc73213dc61a2eb123052dd0fca6f05157', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ramen-noodels/audio_red_round_button_small_val_unnormalized', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ramen-noodels/audio_red_round_button_small_val_unnormalized'), pr_revision=None, pr_num=None)